In [5]:
import pandas as pd
import json
import glob

In [7]:
phrasebank_path = "Sentiment Analysis for Financial News/FinancialPhraseBank/Sentences_75Agree.txt"

phrase_data = []

with open(phrasebank_path, "r", encoding="latin-1") as file:
    for line in file:
        if "@" in line:
            text, sentiment = line.rsplit("@", 1)
            phrase_data.append({
                "text": text.strip(),
                "sentiment": sentiment.strip()
            })

phrasebank_df = pd.DataFrame(phrase_data)

print("PhraseBank Dataset Shape:", phrasebank_df.shape)
phrasebank_df.head()

PhraseBank Dataset Shape: (3453, 2)


,text,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,With the new production plant the company woul...,positive
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,"In the third quarter of 2010 , net sales incre...",positive
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive


In [8]:
import os
import json

json_folder = "Financial News Sentiment Dataset (2012–2022)"

json_files = [
    os.path.join(json_folder, file)
    for file in os.listdir(json_folder)
    if file.endswith(".json")
]

json_data = []

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        json_data.extend(data)

json_df = pd.DataFrame(json_data)

print("JSON Dataset Shape:", json_df.shape)
json_df.head()

JSON Dataset Shape: (209527, 6)


,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/ryan-reynolds-s...,Ryan Reynolds Sums Up The Tragedy Of Losing Be...,ENTERTAINMENT,Reynolds managed to write a beautiful eulogy f...,Elyse Wanshel,2021-12-31
1,https://www.huffpost.com/entry/betty-white-dea...,"Betty White, Beloved 'Golden Girl' Actor, Dead...",ENTERTAINMENT,"The ""Golden Girl"" died on Dec. 31.",Carly Ledbetter,2021-12-31
2,https://www.huffpost.com/entry/bc-eu-britain-p...,Ghislaine Maxwell Verdict Bodes Ill For Prince...,WORLD NEWS,Prince Andrew wasn’t on trial in the Ghislaine...,"DANICA KIRKA, AP",2021-12-31
3,https://www.huffpost.com/entry/funniest-tweets...,26 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Animals are so funny you can name a cat iphon...",Elyse Wanshel,2021-12-31
4,https://www.huffpost.com/entry/skin-care-resol...,"The 5 Best Skin Care Resolutions You Can Make,...",STYLE & BEAUTY,"You already know No. 1, but will you actually ...",Abigail Abesamis Demarest,2021-12-31


In [9]:
print(json_df["category"].unique())

['ENTERTAINMENT' 'WORLD NEWS' 'COMEDY' 'STYLE & BEAUTY' 'POLITICS'
 'ENVIRONMENT' 'FOOD & DRINK' 'U.S. NEWS' 'HOME & LIVING' 'CRIME'
 'WELLNESS' 'BUSINESS' 'SCIENCE' 'SPORTS' 'TRAVEL' 'PARENTING' 'WOMEN'
 'WEIRD NEWS' 'BLACK VOICES' 'MEDIA' 'TECH' 'MONEY' 'RELIGION' 'EDUCATION'
 'CULTURE & ARTS' 'QUEER VOICES' 'LATINO VOICES' 'TASTE' 'GREEN' 'IMPACT'
 'COLLEGE' 'HEALTHY LIVING' 'STYLE' 'PARENTS' 'ARTS & CULTURE'
 'THE WORLDPOST' 'GOOD NEWS' 'WORLDPOST' 'FIFTY' 'ARTS' 'WEDDINGS'
 'DIVORCE']


In [10]:
financial_categories = ["BUSINESS", "MONEY"]

filtered_json_df = json_df[
    json_df["category"].isin(financial_categories)
].copy()

print("Filtered Financial News Shape:", filtered_json_df.shape)
filtered_json_df.head()

Filtered Financial News Shape: (7748, 6)


,link,headline,category,short_description,authors,date
32,https://www.huffpost.com/entry/holiday-sales-r...,Holiday Sales Rise Despite Supply Issues And O...,BUSINESS,"After omicron hit, some consumers stayed home ...","Paul Wiseman and Anne D'Innocenzio, AP",2021-12-26
70,https://www.huffpost.com/entry/ransomware-atta...,Low-Profile Ransomware Attacks Continue As Hig...,BUSINESS,"Cyberattacks have interrupted schools, hospita...","Eric Tucker and Alan Suderman Washington, AP",2021-12-18
197,https://www.huffpost.com/entry/bc-us-black-fri...,Stores Kick Off Black Friday But Pandemic Woes...,BUSINESS,Retailers are expected to usher in the unoffic...,"ANNE D'INNOCENZIO, AP",2021-11-26
398,https://www.huffpost.com/entry/holiday-shoppin...,Why You Should Get Your Holiday Shopping Done ...,MONEY,"Supply chain issues are out of your control, b...",Caroline Bologna,2021-10-21
399,https://www.huffpost.com/entry/flight-attendan...,Flight Attendants At American Airlines Subsidi...,BUSINESS,Workers at Piedmont Airlines say their wages a...,Dave Jamieson,2021-10-21


In [11]:
filtered_json_df["text"] = (
    filtered_json_df["headline"].fillna("") + " " +
    filtered_json_df["short_description"].fillna("")
)

filtered_json_df = filtered_json_df[["text"]]

print("Structured JSON Dataset Shape:", filtered_json_df.shape)
filtered_json_df.head()

Structured JSON Dataset Shape: (7748, 1)


,text
32,Holiday Sales Rise Despite Supply Issues And O...
70,Low-Profile Ransomware Attacks Continue As Hig...
197,Stores Kick Off Black Friday But Pandemic Woes...
398,Why You Should Get Your Holiday Shopping Done ...
399,Flight Attendants At American Airlines Subsidi...


In [12]:
filtered_json_df["sentiment"] = "unlabeled"

In [13]:
combined_df = pd.concat(
    [phrasebank_df, filtered_json_df],
    ignore_index=True
)

print("Combined Dataset Shape:", combined_df.shape)
combined_df.head()

Combined Dataset Shape: (11201, 2)


,text,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,With the new production plant the company woul...,positive
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,"In the third quarter of 2010 , net sales incre...",positive
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive


In [14]:
combined_df.drop_duplicates(subset="text", inplace=True)

print("After Removing Duplicates:", combined_df.shape)

After Removing Duplicates: (11194, 2)


In [15]:
combined_df.to_csv("combined_financial_dataset.csv", index=False)

print("Dataset saved as combined_financial_dataset.csv")

Dataset saved as combined_financial_dataset.csv
